# Banyan City — free anime rendering on Kaggle

Renders a node's `shots.md` prompts into per-beat clips: **SDXL** still then **SVD** motion, on an anime-tuned
**SD1.5** checkpoint, on Kaggle's free GPU quota (30 h/week) — the tree's permanent $0 rendering
floor, reproducible by any citizen (**compute-as-watering**, see `WATERING.md`).

**Setup:** Kaggle → New Notebook → File → Import Notebook → this file. Settings: Accelerator =
**GPU T4 x2**, Internet = ON, phone-verified account. Or drive it headless from a laptop with
`python3 pipeline/kaggle/run_remote.py push <node>`, which is the only mode whose output can be
retrieved — an interactive session dies with the browser tab and `kernels output` 404s.

**Why not Wan 2.1** (tried first, 2026-07-25/26, six pushes): Wan is trained in **bfloat16**, and no
Kaggle free accelerator supports bf16 — T4 is Turing sm_75, P100 is Pascal sm_60, and bf16 arrives
with Ampere sm_80. In fp16 its activations overflow to NaN and every frame decodes to flat grey, at
29 minutes a shot. fp32 is numerically safe and ~8x slower, which cannot finish an episode inside a
12-hour session. AnimateDiff on SD1.5 is fp16-native, runs in minutes, and an anime checkpoint is a
better match for `style.md`'s flat cel-shaded look than a general-purpose video model.

**Hard-won details, each of which cost a run:**
- The repo is cloned to `/kaggle/tmp`, NOT `/kaggle/working`. Everything in `/kaggle/working` becomes
  the session's published output, and Kaggle caps how many files it indexes — a checkout there
  crowded the actual clips out of the output entirely.
- `machine_shape: NvidiaTeslaT4` is set in `kernel-metadata.json`. Without it the batch scheduler
  hands out a P100, which current torch ships no kernels for at all.
- The setup cell defines `transformers.utils.FLAX_WEIGHTS_NAME`, which the batch image's newer
  transformers removed and diffusers 0.33 imports at module load.
- Every clip's middle frame is checked for contrast before it is written, and the session ABORTS on a
  blank one. A numerically dead generation still writes a valid mp4 that passes every container,
  duration and audio check.

Output: `/kaggle/working/clips.zip`, re-zipped after every clip → feed to
`pipeline/render_t3.py <genome> <node> --clips <dir>`. Clips are short (3s at 8fps); `render_t3`
ping-pong-loops them to fill a beat, so a beat never shows a hard loop seam.

Provenance: every clip gets a `meta.yaml` (§7.2). The season's canon quality bar is decided by the
founder on material (R4/D8).


In [ ]:
# ---- config: what to render ----------------------------------------------
GENOME = "sapling"
# Founder-review gate (loop.md cadence): stills are the unit of review. A stills-only
# run gives the founder all 15 frames in ~10 min of GPU instead of ~70, and NOTHING is
# animated until the frame under it is approved.
MOTION_ENGINE = "ltx"
STILLS_ONLY = False
# Pick-of-N: draw this many seed-variants per beat. Prompt-tweaking an out-of-
# distribution composition is a coin flip (three founder-rejected rounds prove it);
# variants turn one flip per round into four, and the founder points at the winner.
SEEDS_PER_BEAT = 1
NODE   = "001-capability-inventory"        # any node id with a shots.md
BEATS  = [1]          # e.g. [1, 3] or None for all beats without status ✅
SEED   = 20260719      # fixed base seed: beat N renders with SEED + N (reproducible)
FPS_OUT = 7            # SVD is conditioned at 7fps
STEPS  = 40            # 30 = faster/rougher, 50 = slower/cleaner
# SDXL-class stills. SD1.5 was only ever here because AnimateDiff REQUIRED it, and
# that requirement died with AnimateDiff — SVD is image-to-video and does not care
# what drew the frame. Founder, 2026-07-27: "the image generator you are using also
# looks like ai 5 years ago" — fair: SD1.5 shipped in 2022, and I never revisited the
# choice after removing the reason for it. Anime-tuned SDXL first, then base SDXL.
BASES  = ["cagliostrolab/animagine-xl-3.1",
          "stabilityai/stable-diffusion-xl-base-1.0"]
IPA_SDXL = "ip-adapter_sdxl.bin"   # pairs with the image encoder diffusers loads from
                                   # sdxl_models/. The _vit-h variant wants a DIFFERENT
                                   # encoder and fails with "mat1 and mat2 shapes cannot
                                   # be multiplied (2x1280 and 1024x8192)".
SVD    = "stabilityai/stable-video-diffusion-img2vid-xt"  # stage 2: real motion
STILL_W, STILL_H = 832, 1216  # SDXL's portrait training bucket — the still's quality
SVD_W, SVD_H = 512, 768       # what SVD can afford on a 14.56 GiB T4; the still is
                              # downscaled to this before animating, and render_t3
                              # scales the result to 720x1280 on assembly anyway
MOTION = 160           # SVD motion_bucket_id: 20 = almost still, 180 = a lot.
                       # 127 gave MOTION 0.3 on a sparse still — SVD animates image
                       # CONTENT, so a subject on an empty ground has nothing to move.
SVD_FRAMES = 25        # 25 @ 7fps = ~3.5s of ACTUAL movement
IPADAPTER = True       # condition recurring characters on genomes/<g>/refs/*.png
# One image that defines the show's LOOK, conditioned into every beat that has no
# character reference of its own. The founder's objection to the first cut was that
# it read as "fifteen unrelated AI images"; a shared plate is the direct answer, and
# which frame *is* the show is his call (R4), not the renderer's. Drop a file at
# genomes/<genome>/refs/style-plate.png and every beat inherits it — no code change.
# Lower than IPA_SCALE on purpose: this must tint the palette, not dictate content.
STYLE_PLATE = "style-plate.png"
STYLE_SCALE = 0.30
IPA_SCALE = 0.35       # identity only. At 0.6 the reference dictated COMPOSITION too:
                       # four different beats — including a sprint and a close-up — all came
                       # back as the same centred standing figure in a circular vignette, and
                       # contrast dropped enough to trip the blank guard on two of them.
REPO_URL = "https://github.com/olegmlkvorg/banyan-city.git"


In [ ]:
# ---- setup: deps + repo (canon prompts come from shots.md, not a paste) ---
# Do NOT reinstall torch. Kaggle ships a working torch/torchvision pair; the
# first real run (2026-07-25) tried to pin its own and hit INTERNAL ASSERT
# FAILED in Dtype.cpp — a fresh torchvision against the already-imported
# torch. Install diffusers with --no-deps so pip cannot pull a second torch in
# behind it. WanPipeline needs diffusers >= 0.33 (0.32 lacks it entirely —
# that was the failure after the pin).
%pip -q install --no-deps "diffusers==0.33.1"
%pip -q install ftfy imageio imageio-ffmpeg pyyaml psutil

# A kernel that already imported an older diffusers keeps it in memory no
# matter what pip writes to disk (this bit the founder on 2026-07-25: the
# session still held 0.32.2). Compare disk vs memory and say so plainly.
import sys
from importlib.metadata import version
on_disk = version("diffusers")
in_mem = getattr(sys.modules.get("diffusers"), "__version__", None)
print(f"diffusers on disk: {on_disk}" + (f" | already imported in this kernel: {in_mem}" if in_mem else ""))
if in_mem and in_mem != on_disk:
    print("\n*** STOP: this kernel is holding an older diffusers.\n"
          "    Run > Restart & clear cell outputs, then Run All again.\n"
          "    (Nothing is lost — finished clips are skipped on re-run.) ***\n")


# Kaggle's BATCH image ships a newer transformers than its interactive one, and
# `transformers.utils.FLAX_WEIGHTS_NAME` is gone from it. diffusers 0.33 imports
# that name at module load, so `from diffusers import WanPipeline` died with
# "cannot import name 'FLAX_WEIGHTS_NAME'" before touching the GPU (first batch
# push, 2026-07-25). The names are plain filename constants and nothing on the
# Wan path reads a flax/tf checkpoint, so define what is missing rather than
# repinning transformers — a repin drags tokenizers and risks the torch pair.
# diffusers 0.33 also references transformers.CLIPFeatureExtractor, which current
# transformers renamed to CLIPImageProcessor. That is what made Lykon/dreamshaper-8
# — the anime-capable candidate — fail to load with "module transformers has no
# attribute CLIPFeatureExtractor", leaving only vanilla SD1.5, whose house style is
# watercolour rather than cel-shaded anime. Alias it.
import transformers as _tf
if not hasattr(_tf, "CLIPFeatureExtractor") and hasattr(_tf, "CLIPImageProcessor"):
    _tf.CLIPFeatureExtractor = _tf.CLIPImageProcessor
    print("aliased transformers.CLIPFeatureExtractor -> CLIPImageProcessor")

import transformers.utils as _tu
for _name, _val in (("FLAX_WEIGHTS_NAME", "flax_model.msgpack"),
                    ("TF2_WEIGHTS_NAME", "tf_model.h5"),
                    ("TF_WEIGHTS_NAME", "model.ckpt")):
    if not hasattr(_tu, _name):
        setattr(_tu, _name, _val)
        print(f"shimmed transformers.utils.{_name} (removed upstream)")

import pathlib
import subprocess
import sys

# Clone OUTSIDE /kaggle/working. Everything in /kaggle/working becomes the
# session's downloadable output, so cloning the repo there put the whole
# checkout — every committed episode mp4 included — into the output: the first
# successful run's `kernels output` was still pulling at 755 MB and had not
# reached the one clip that was actually rendered (2026-07-25). /kaggle/tmp is
# scratch and is not published.
CHECKOUT = pathlib.Path("/kaggle/tmp/banyan-city")
if not CHECKOUT.exists():
    CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CHECKOUT)], check=True)
sys.path.insert(0, str(CHECKOUT / "pipeline"))
from generate_shots import parse_shots
from sd_prompt import compress, extra_negatives, suppressed_negatives  # CLIP stops at 77 tokens
import yaml

node_dirs = [d for d in (CHECKOUT / "genomes" / GENOME / "nodes").iterdir() if d.is_dir()]
node_dir = next((d for d in sorted(node_dirs) if d.name.startswith(NODE)), None)
assert node_dir, f"no node dir starting with {NODE!r} — check NODE above"
shots = parse_shots((node_dir / "shots.md").read_text())
todo = [s for s in shots if (BEATS is None and not s["done"]) or (BEATS and s["num"] in BEATS)]
print(f"{len(todo)} beat(s) to render for {node_dir.name}:")
for s in todo:
    print(f"  {s['num']:02d} {s['slug']}")

# the negative prompt is defined here so the model cell's bisect can use it too
# "abstract" is here because Animagine XL 3.1's own model card lists it, and on
# 2026-07-26 leaving it out returned beats 3 and 4 as literal abstract shapes.
NEG = ("photorealistic, 3d render, abstract, text, watermark, signature, low quality, "
       "blurry, extra limbs, deformed, jpeg artifacts, realistic skin texture")


In [ ]:
# ---- stage 1 model: SD1.5 makes the PICTURE ----------------------------------
# AnimateDiff was abandoned on the founder's verdict, 2026-07-26: "pretty much
# static, you could say. just cool looking static". He was right, and the number
# was already in front of me — frame-to-frame change measured 0.007-0.077, which I
# had read as "coherent video, not noise" without ever asking whether anything
# MOVED. The v1.5 motion module drifts; it does not animate. 16 frames at 8fps,
# ping-pong-looped to fill a five-second beat, is a shimmering still.
#
# So the job splits along the line of what actually works. SD1.5 makes a good
# still: measured contrast 57 on beat 1, on-style, reliable, and IP-Adapter keeps a
# character looking like himself. Stable Video Diffusion is image-to-video — hand
# it that still and it generates real camera and subject motion. It runs in fp16,
# so no bf16 wall, and it is free.
#
# One model resident at a time: all the stills first, then the SD pipeline is
# released and SVD is loaded. Two pipelines at once does not fit 14.6 GiB.
import gc

import numpy as np
import psutil
import torch
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator = GPU (needs phone verification)"
_cap = torch.cuda.get_device_capability(0)
if f"sm_{_cap[0]}{_cap[1]}" not in torch.cuda.get_arch_list():
    raise SystemExit(f"{torch.cuda.get_device_name(0)} is sm_{_cap[0]}{_cap[1]}; this torch "
                     f"was built for {torch.cuda.get_arch_list()}. Use GPU T4 x2.")

# NOT AutoPipelineForText2Image: it eagerly imports every pipeline in the registry,
# including HunyuanDiT, which needs an MT5Tokenizer that current transformers has
# removed — so importing it fails with an error naming neither SDXL nor us. Import
# the one pipeline we want.
from diffusers import DDIMScheduler, StableDiffusionXLPipeline
from PIL import Image

print(f"{torch.cuda.get_device_name(0)}: "
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB VRAM | "
      f"{psutil.virtual_memory().available / 2**30:.1f} GiB RAM")


def luma_spread(img):
    """LUMA spread 0-255. Must be luma: RGB percentiles are inflated by colour."""
    a = np.asarray(img, dtype=np.float32)
    if not np.isfinite(a).all():
        return 0.0
    if a.ndim == 3 and a.shape[-1] >= 3:
        a = a[..., 0] * 0.299 + a[..., 1] * 0.587 + a[..., 2] * 0.114
    lo, hi = np.percentile(a, 10), np.percentile(a, 90)
    return float(hi - lo) * (255.0 if a.max() <= 1.001 else 1.0)


def motion_of(frames):
    """Mean absolute frame-to-frame change, 0-255. The measure I failed to take.

    AnimateDiff's output scored near zero here while passing every contrast check I
    owned, which is exactly how "cool looking static" got shipped as success."""
    g = [np.asarray(f.convert("L"), dtype=np.float32) for f in frames]
    if len(g) < 2:
        return 0.0
    return float(np.mean([np.abs(g[i + 1] - g[i]).mean() for i in range(len(g) - 1)]))


# The checkpoint's own scheduler is kept — forcing a schedule is what produced
# washed, unresolved frames earlier today.
sd = BASE = None
for cand in BASES:
    for kw in ({"variant": "fp16"}, {}):     # some repos ship no fp16 variant
        try:
            sd = StableDiffusionXLPipeline.from_pretrained(
                cand, torch_dtype=torch.float16, use_safetensors=True,
                add_watermarker=False, **kw)
            BASE = cand
            break
        except Exception as e:
            print(f"  {cand} {kw or 'no-variant'} unavailable "
                  f"({type(e).__name__}: {str(e)[:70]})", flush=True)
    if sd is not None:
        break
if sd is None:
    raise SystemExit(f"none of {BASES} could be loaded")
# Only attach the adapter if a beat in THIS run actually features a referenced
# character. Most beats have none — 001 has no goblin at all — and the character-free
# path still had to push a neutral image through the encoder, which meant every
# character-free render carried the adapter's memory cost and its failure modes for
# no benefit.
REFS = {}
_rdir = CHECKOUT / "genomes" / GENOME / "refs"
_names = [f.stem.lower() for f in sorted(_rdir.glob("*.png"))] if _rdir.is_dir() else []
WHO = {"jerry": ("goblin", "scavenger", "jerry")}
_needed = sorted({n for n in _names if n in WHO
                  and any(w in s["prompt"].lower() for s in todo for w in WHO[n])})
PLATE = None
_plate_f = _rdir / STYLE_PLATE
_want_ipa = bool(_needed) or _plate_f.is_file()
if IPADAPTER and not _want_ipa:
    print(f"ip-adapter: not loaded — no style plate, and no beat in this run features {_names or 'any reference'}")
if IPADAPTER and _want_ipa:
    try:
        _sub = "sdxl_models" if "xl" in BASE.lower() else "models"
        _wt = IPA_SDXL if "xl" in BASE.lower() else "ip-adapter_sd15.bin"
        sd.load_ip_adapter("h94/IP-Adapter", subfolder=_sub, weight_name=_wt)
        sd.set_ip_adapter_scale(IPA_SCALE)
        for f in sorted(_rdir.glob("*.png")):
            if f.stem.lower() in _needed and f.name != STYLE_PLATE:
                REFS[f.stem.lower()] = Image.open(f).convert("RGB").resize((512, 512))
        if _plate_f.is_file():
            PLATE = Image.open(_plate_f).convert("RGB").resize((512, 512))
        print(f"ip-adapter: characters {sorted(REFS)} at {IPA_SCALE}"
              f"; style plate {'yes' if PLATE is not None else 'no'} at {STYLE_SCALE}")
    except Exception as e:
        print(f"ip-adapter unavailable ({type(e).__name__}: {str(e)[:80]})")
        REFS, PLATE = {}, None

# Offload LAST, after IP-Adapter is attached. enable_model_cpu_offload() installs
# hooks on the modules present when it runs, so anything added afterwards — like the
# adapter's image encoder — keeps its weights on the CPU while the pipeline feeds it
# CUDA tensors: "Input type (torch.cuda.HalfTensor) and weight type
# (torch.HalfTensor) should be the same" (2026-07-27). Order is the fix, not a cast.
sd.enable_model_cpu_offload()
for _o in ("enable_vae_slicing", "enable_vae_tiling"):
    _f = getattr(sd, _o, None)
    if callable(_f):
        _f()


# Once an IP-Adapter is loaded, the UNet expects image embeds on EVERY call — pass
# none and it dies with "argument of type 'NoneType' is not iterable" deep inside
# the pipeline (2026-07-26, beat 1, which has no character in it). So a beat with no
# character gets a neutral grey image at scale 0, which conditions on nothing.
NEUTRAL = Image.new("RGB", (512, 512), (128, 128, 128))
gc.collect(); torch.cuda.empty_cache()
print(f"stage 1 ready — {BASE} at {STILL_W}x{STILL_H}")


In [ ]:
# ---- stage 1: a still per beat, then stage 2: SVD animates each one -----------
import shutil
import time
from datetime import date

# "abstract" is here because Animagine XL 3.1's own model card lists it, and on
# 2026-07-26 leaving it out returned beats 3 and 4 as literal abstract shapes.
NEG = ("photorealistic, 3d render, abstract, text, watermark, signature, low quality, "
       "blurry, extra limbs, deformed, jpeg artifacts, realistic skin texture")
out = pathlib.Path("/kaggle/working/clips"); out.mkdir(parents=True, exist_ok=True)
stills = pathlib.Path("/kaggle/working/stills"); stills.mkdir(parents=True, exist_ok=True)
blank, prompts, negs = [], {}, {}

# ---- stage 1 ----------------------------------------------------------------
todo = [dict(s, seed_k=k) for s in todo for k in range(max(1, SEEDS_PER_BEAT))]
for s in todo:
    _v = f"-s{s['seed_k']}" if SEEDS_PER_BEAT > 1 else ""
    png = stills / f"{s['num']:02d}-{s['slug']}{_v}.png"
    # A founder-APPROVED still committed to the repo is canonical: use those exact
    # pixels rather than redrawing. Approval binds pixels, not prompts — a redraw,
    # even from the same prompt and seed, is not what was approved.
    _approved = node_dir / "stills" / png.name
    if not png.exists() and _approved.exists():
        import shutil; shutil.copy(_approved, png)
        print(f"approved still {png.name} from repo — not redrawing", flush=True)
    if png.exists():
        print(f"skip still {png.name}"); prompts[s["num"]] = compress(s["prompt"])[0]
        negs[s["num"]] = NEG; continue
    t0 = time.time()
    ptext, dropped = compress(s["prompt"])
    prompts[s["num"]] = ptext
    extra = extra_negatives(s["prompt"])
    neg = f"{NEG}, {extra}" if extra else NEG
    # A beat whose SUBJECT is a screen cannot have "text" in its negative prompt.
    for _drop in suppressed_negatives(s["prompt"]):
        neg = neg.replace(_drop + ", ", "")
    negs[s["num"]] = neg          # provenance: record what was actually sent (S7.2)
    ref = None
    for _n, _w in WHO.items():
        if _n in REFS and any(w in s["prompt"].lower() for w in _w):
            ref = REFS[_n]; break
    print(f"still {s['num']:02d} ({s['slug']}) …", flush=True)
    print(f"   {ptext}", flush=True)
    kw = {}
    if REFS or PLATE is not None:
        # A beat with a character reference uses it; every other beat falls back to the
        # style plate so the episode reads as one show. NEUTRAL + scale 0 stays the
        # no-op path when there is neither: the adapter is loaded, so an image is
        # mandatory even when we want it to have no effect.
        _img, _scale = (ref, IPA_SCALE) if ref is not None else (PLATE, STYLE_SCALE)
        sd.set_ip_adapter_scale(_scale if _img is not None else 0.0)
        kw["ip_adapter_image"] = _img if _img is not None else NEUTRAL
    img = sd(prompt=ptext, negative_prompt=neg, height=STILL_H, width=STILL_W,
             num_inference_steps=STEPS, guidance_scale=7.5,
             generator=torch.Generator(device="cpu").manual_seed(SEED + s["num"] + s.get("seed_k", 0) * 1000),
             **kw).images[0]
    sp = luma_spread(img)
    img.save(png)
    print(f"   {png.name} in {(time.time()-t0)/60:.1f} min, contrast {sp:.0f}"
          + (f"  ref:{[n for n in WHO if REFS.get(n) is ref]}" if ref is not None
             else ("  ref:style-plate" if PLATE is not None else "")),
          flush=True)
    if sp < 35:
        blank.append((s["num"], s["slug"], round(sp)))
        print("   BLANK still — will still be animated, but flagged", flush=True)

# Releasing the SDXL pipeline takes more than `del`. enable_model_cpu_offload
# installs accelerate hooks that keep GPU allocations alive, so after `del sd` the
# card still held ~12.7 of 14.56 GiB and SVD could not get 1.88 GiB (2026-07-27).
# maybe_free_model_hooks() is what actually detaches them.
try:
    sd.maybe_free_model_hooks()
except Exception:
    pass
for _attr in ("unet", "vae", "text_encoder", "text_encoder_2", "image_encoder"):
    _m = getattr(sd, _attr, None)
    if _m is not None:
        try:
            _m.to("cpu")
        except Exception:
            pass
del sd
gc.collect(); torch.cuda.empty_cache()
_free, _total = torch.cuda.mem_get_info()
print(f"\nstage 1 done: {len(list(stills.glob('*.png')))} still(s); "
      f"{_free/2**30:.1f} of {_total/2**30:.1f} GiB VRAM free for stage 2\n")

# ---- stage 2: real motion ---------------------------------------------------
if STILLS_ONLY:
    print("STILLS_ONLY — stage 2 skipped; stills await the founder's verdicts")
else:
    from diffusers import StableVideoDiffusionPipeline
    from diffusers.utils import export_to_video

    if MOTION_ENGINE == "ltx":
        from diffusers import LTXImageToVideoPipeline
        from diffusers.utils import export_to_video, load_image
        ltx = LTXImageToVideoPipeline.from_pretrained(
            "Lightricks/LTX-Video", torch_dtype=torch.float16)
        ltx.enable_model_cpu_offload()
        print("stage 2 ready - LTX-Video fp16", flush=True)
        for s in todo:
            png = stills / f"{s['num']:02d}-{s['slug']}.png"
            dest = out / f"{s['num']:02d}-{s['slug']}.mp4"
            if not png.exists() or dest.exists():
                continue
            t0 = time.time()
            print(f"ltx animate {s['num']:02d} ({s['slug']}) ...", flush=True)
            image = load_image(str(png)).resize((512, 768))
            _mp = prompts.get(s["num"]) or compress(s["prompt"])[0]
            frames = ltx(image=image, prompt=_mp,
                         negative_prompt="worst quality, inconsistent motion, blurry, jittery, distorted",
                         width=512, height=768, num_frames=97,
                         num_inference_steps=30,
                         generator=torch.Generator(device="cpu").manual_seed(SEED + s["num"])).frames[0]
            export_to_video(frames, str(dest), fps=24)
            _sp = sorted(luma_spread(f) for f in frames[::12])
            print(f"   {dest.name} in {(time.time()-t0)/60:.1f} min, "
                  f"contrast {_sp[len(_sp)//2]:.0f}", flush=True)
            dest.with_suffix(".meta.yaml").write_text(
                "# Shot provenance (7.2)\n" + yaml.safe_dump({
                    "platform": "kaggle-free-gpu",
                    "model": f"still: {BASE} | motion: Lightricks/LTX-Video",
                    "prompt": _mp, "seed": SEED + s["num"],
                    "frames": 97, "fps": 24, "cost_usd": 0}))
            import shutil as _sh
            _sh.make_archive("/kaggle/working/clips", "zip", out)
    else:
        svd = StableVideoDiffusionPipeline.from_pretrained(SVD, torch_dtype=torch.float16,
                                                          variant="fp16")
        # SVD is the larger model; offload rather than hold it resident on a T4
        svd.enable_model_cpu_offload()
        # Ask before calling: SVD's VAE is AutoencoderKLTemporalDecoder, which has no
        # enable_slicing (unlike the SD/SDXL VAEs). I feature-detected exactly this for
        # Wan's VAE earlier today and then hardcoded the call here.
        for _opt in ("enable_slicing", "enable_tiling"):
            _fn = getattr(svd.vae, _opt, None)
            if callable(_fn):
                _fn(); print(f"svd vae: {_opt}()")
        print(f"stage 2 ready — {SVD}, motion_bucket={MOTION}, {SVD_FRAMES} frames @ {FPS_OUT}fps\n")

        still_motion = []
        for s in todo:
            png = stills / f"{s['num']:02d}-{s['slug']}.png"
            dest = out / f"{s['num']:02d}-{s['slug']}.mp4"
            if not png.exists() or dest.exists():
                continue
            t0 = time.time()
            print(f"animate {s['num']:02d} ({s['slug']}) …", flush=True)
            # SVD is animated at a smaller size than the still. SDXL draws 832x1216 for
            # quality, but SVD at that size wants ~1.9 GiB per step on a 14.56 GiB card and
            # OOMs; 512x768 is what it ran comfortably at all evening, and render_t3 scales
            # to 720x1280 on assembly regardless.
            img = Image.open(png).convert("RGB").resize((SVD_W, SVD_H), Image.LANCZOS)
            try:
                # SVD defaults to its training size, 1024x576 LANDSCAPE, and silently
                # ignores the aspect of the still you hand it — beat 1 came back widescreen
                # from a 512x768 portrait input (2026-07-26), and cropping that to 9:16
                # would throw away most of the frame. Ask for portrait explicitly.
                frames = svd(img, height=SVD_H, width=SVD_W,
                             num_frames=SVD_FRAMES, motion_bucket_id=MOTION,
                             noise_aug_strength=0.08, decode_chunk_size=4,   # more noise = more movement
                             generator=torch.Generator(device="cpu").manual_seed(SEED + s["num"] + s.get("seed_k", 0) * 1000)
                             ).frames[0]
            except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                if not isinstance(e, torch.cuda.OutOfMemoryError) and \
                   not any(k in str(e) for k in ("CUBLAS", "out of memory", "CUDA error")):
                    raise
                print(f"   {type(e).__name__} — retrying with fewer frames", flush=True)
                gc.collect(); torch.cuda.empty_cache()
                frames = svd(img, height=SVD_H, width=SVD_W,
                             num_frames=14, motion_bucket_id=MOTION,
                             noise_aug_strength=0.02, decode_chunk_size=2,
                             generator=torch.Generator(device="cpu").manual_seed(SEED + s["num"] + s.get("seed_k", 0) * 1000)
                             ).frames[0]
            mo, sp = motion_of(frames), float(np.median([luma_spread(f) for f in frames[::4]]))
            export_to_video(frames, str(dest), fps=FPS_OUT)
            shutil.make_archive("/kaggle/working/clips", "zip", out)
            still_motion.append((s["num"], round(mo, 1)))
            print(f"   {dest.name} in {(time.time()-t0)/60:.1f} min, "
                  f"contrast {sp:.0f}, MOTION {mo:.1f}", flush=True)
            dest.with_suffix(".meta.yaml").write_text(
                "# Shot provenance (\u00a77.2)\n" + yaml.safe_dump({
                    "platform": "kaggle-free-gpu",
                    "model": f"still: {BASE} (+IP-Adapter {IPA_SCALE}) | motion: {SVD}",
                    "prompt": prompts.get(s["num"], ""), "negative_prompt": negs.get(s["num"], NEG),
                    "seed": SEED + s["num"], "steps": STEPS,
                    "frames": len(frames), "fps": FPS_OUT, "motion_bucket_id": MOTION,
                    "measured_motion": round(mo, 2), "cost_usd": 0.00,
                    "generated": str(date.today()),
                }, sort_keys=False))

        # Motion is REPORTED, not gated: no threshold has been calibrated yet, and a
        # mis-calibrated guard cost most of 2026-07-26. AnimateDiff scored near zero here.
        if still_motion:
            print("\nmotion per beat (mean frame-to-frame change, 0-255):")
            for n, mo in still_motion:
                print(f"  beat {n:02d}: {mo}")
            print(f"  median {np.median([m for _, m in still_motion]):.1f} — "
                  "AnimateDiff measured ~0.1-1.0 on the same beats")
        if blank:
            print(f"\n{len(blank)} still(s) came out flat: {blank}")


In [ ]:
# ---- pack for download ------------------------------------------------------
import shutil
shutil.make_archive("/kaggle/working/clips", "zip", "/kaggle/working/clips")
print("download clips.zip from the Output tab, then locally:")
print(f"  python3 pipeline/render_t3.py {GENOME} {NODE} --clips <unzipped-dir> --out /tmp/{NODE}-episode.mp4")